# 🧠 K'UHUL Training Server for Colab

This notebook sets up a training server that connects to your browser-based K'UHUL environment.

## Features
- 🚀 Free GPU training (Tesla T4/P100/V100)
- 📊 Real-time metrics streaming
- 💾 Model checkpoint management
- 🔄 Automatic sync with browser
- 🎨 SVG weight export

## Setup Instructions
1. Run **Setup** cell to install dependencies
2. Run **Start Server** cell to launch API
3. Copy the **ngrok URL** to your browser
4. Submit training jobs from the dashboard!


In [ ]:
# 📦 CELL 1: Install Dependencies
print('🔧 Installing K\'UHUL dependencies...')

!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets
!pip install -q flask flask-cors
!pip install -q pyngrok
!pip install -q bitsandbytes accelerate
!pip install -q tensorboard wandb

print('✅ Dependencies installed!')

In [ ]:
# 🔍 CELL 2: Check GPU
import torch
print('🖥️  GPU Information:')
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'CUDA Version: {torch.version.cuda}')
else:
    print('⚠️  No GPU detected. Enable GPU in Runtime > Change runtime type')

In [ ]:
# 🏗️ CELL 3: K'UHUL Training Server
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import time
import json
from datetime import datetime

app = Flask(__name__)
CORS(app)

# Global state
sessions = {}
training_jobs = {}
datasets = {}

class TrainingJob:
    def __init__(self, job_id, config):
        self.id = job_id
        self.config = config
        self.status = 'queued'
        self.progress = 0
        self.epoch = 0
        self.metrics = {'loss': [], 'accuracy': []}
        self.started_at = None
        self.model = None
        
    def run(self):
        """Simulate training (replace with actual training logic)"""
        self.status = 'running'
        self.started_at = datetime.now()
        
        total_epochs = self.config.get('epochs', 100)
        
        for epoch in range(total_epochs):
            if self.status == 'cancelled':
                break
                
            # Simulate training step
            time.sleep(0.5)  # Simulate computation
            
            # Update metrics
            loss = 2.0 * (1 - epoch / total_epochs) + 0.1
            accuracy = 0.5 + 0.5 * (epoch / total_epochs)
            
            self.metrics['loss'].append(loss)
            self.metrics['accuracy'].append(accuracy)
            self.epoch = epoch + 1
            self.progress = (epoch + 1) / total_epochs * 100
            
        if self.status != 'cancelled':
            self.status = 'completed'
            self.progress = 100

# API Endpoints

@app.route('/api/ping', methods=['POST'])
def ping():
    data = request.json
    session_id = data.get('sessionId')
    sessions[session_id] = {'connected_at': datetime.now()}
    return jsonify({'status': 'ok', 'timestamp': time.time()})

@app.route('/api/disconnect', methods=['POST'])
def disconnect():
    data = request.json
    session_id = data.get('sessionId')
    if session_id in sessions:
        del sessions[session_id]
    return jsonify({'success': True})

@app.route('/api/training/submit', methods=['POST'])
def submit_training():
    data = request.json
    job_data = data.get('job')
    
    job_id = f"colab_{int(time.time() * 1000)}"
    job = TrainingJob(job_id, job_data.get('config', {}))
    training_jobs[job_id] = job
    
    # Start training in background thread
    thread = threading.Thread(target=job.run)
    thread.start()
    
    return jsonify({
        'success': True,
        'jobId': job_id,
        'status': 'queued'
    })

@app.route('/api/training/status', methods=['POST'])
def get_status():
    data = request.json
    job_id = data.get('jobId')
    
    if job_id not in training_jobs:
        return jsonify({'success': False, 'error': 'Job not found'})
    
    job = training_jobs[job_id]
    
    return jsonify({
        'success': True,
        'status': job.status,
        'progress': job.progress,
        'metrics': {
            'loss': job.metrics['loss'],
            'accuracy': job.metrics['accuracy'],
            'epoch': job.epoch
        }
    })

@app.route('/api/training/cancel', methods=['POST'])
def cancel_training():
    data = request.json
    job_id = data.get('jobId')
    
    if job_id in training_jobs:
        training_jobs[job_id].status = 'cancelled'
        return jsonify({'success': True})
    
    return jsonify({'success': False, 'error': 'Job not found'})

@app.route('/api/model/download', methods=['POST'])
def download_model():
    data = request.json
    job_id = data.get('jobId')
    format_type = data.get('format', 'pytorch')
    
    if job_id not in training_jobs:
        return jsonify({'success': False, 'error': 'Job not found'})
    
    job = training_jobs[job_id]
    
    # Simulate model data
    return jsonify({
        'success': True,
        'modelData': 'base64_encoded_model_data',
        'weights': {},
        'metadata': {
            'architecture': 'transformer',
            'parameters': 125000000,
            'final_loss': job.metrics['loss'][-1] if job.metrics['loss'] else 0,
            'final_accuracy': job.metrics['accuracy'][-1] if job.metrics['accuracy'] else 0
        }
    })

@app.route('/api/dataset/upload', methods=['POST'])
def upload_dataset():
    data = request.json
    dataset_name = data.get('datasetName')
    chunk_index = data.get('chunkIndex', 0)
    total_chunks = data.get('totalChunks', 1)
    
    if dataset_name not in datasets:
        datasets[dataset_name] = {'chunks': [], 'metadata': data.get('metadata')}
    
    datasets[dataset_name]['chunks'].append(data.get('chunk'))
    
    return jsonify({'success': True})

@app.route('/api/resources/gpu', methods=['POST'])
def get_gpu_info():
    if not torch.cuda.is_available():
        return jsonify({
            'success': False,
            'error': 'No GPU available'
        })
    
    gpu_props = torch.cuda.get_device_properties(0)
    mem_allocated = torch.cuda.memory_allocated(0) / 1e6
    mem_total = gpu_props.total_memory / 1e6
    
    return jsonify({
        'success': True,
        'gpus': [{
            'name': torch.cuda.get_device_name(0),
            'memory': mem_total,
            'utilization': mem_allocated / mem_total
        }],
        'memory': {
            'total': mem_total,
            'used': mem_allocated,
            'free': mem_total - mem_allocated
        },
        'utilization': mem_allocated / mem_total
    })

print('✅ K\'UHUL Training Server configured')
print('Ready to start!')

In [ ]:
# 🚀 CELL 4: Start Server with ngrok
from pyngrok import ngrok
import threading

# Set your ngrok auth token (get one free at ngrok.com)
# Uncomment and set your token:
# ngrok.set_auth_token('YOUR_NGROK_TOKEN_HERE')

# Start Flask in background thread
def run_flask():
    app.run(port=5000)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Give Flask time to start
time.sleep(2)

# Create ngrok tunnel
public_url = ngrok.connect(5000)

print('\n' + '='*60)
print('🎉 K\'UHUL Training Server is LIVE!')
print('='*60)
print(f'\n📡 Public URL: {public_url}')
print('\n📋 Copy this URL to your browser\'s Colab Bridge connection dialog')
print('\n⚡ GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('\n🔄 Server is running... Keep this cell active!')
print('='*60 + '\n')

# Keep the cell running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print('\n🛑 Server stopped')

## 📊 Monitor Training

The cells below let you monitor training jobs directly in Colab.

In [ ]:
# 📊 CELL 5: View Active Jobs
import pandas as pd

if training_jobs:
    jobs_data = []
    for job_id, job in training_jobs.items():
        jobs_data.append({
            'Job ID': job_id,
            'Status': job.status,
            'Progress': f"{job.progress:.1f}%",
            'Epoch': job.epoch,
            'Loss': f"{job.metrics['loss'][-1]:.4f}" if job.metrics['loss'] else 'N/A',
            'Accuracy': f"{job.metrics['accuracy'][-1]:.4f}" if job.metrics['accuracy'] else 'N/A'
        })
    
    df = pd.DataFrame(jobs_data)
    print('\n🏃 Active Training Jobs:\n')
    print(df.to_string(index=False))
else:
    print('\n📭 No active training jobs')

In [ ]:
# 📈 CELL 6: Plot Training Metrics
import matplotlib.pyplot as plt

job_id = list(training_jobs.keys())[0] if training_jobs else None

if job_id:
    job = training_jobs[job_id]
    
    if job.metrics['loss']:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        
        # Loss plot
        ax1.plot(job.metrics['loss'], color='#16f2aa', linewidth=2)
        ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.grid(alpha=0.3)
        
        # Accuracy plot
        ax2.plot(job.metrics['accuracy'], color='#3b82f6', linewidth=2)
        ax2.set_title('Training Accuracy', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print('⏳ No metrics available yet')
else:
    print('📭 No training jobs to plot')

## 🔧 Advanced Configuration

Customize training behavior below.

In [ ]:
# ⚙️ CELL 7: Configuration
CONFIG = {
    'max_concurrent_jobs': 3,
    'checkpoint_interval': 10,  # Save every N epochs
    'enable_wandb': False,
    'enable_tensorboard': False,
    'mixed_precision': True,
    'gradient_checkpointing': True
}

print('⚙️  Configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

## 📚 Documentation

**API Endpoints:**
- `POST /api/ping` - Check connection
- `POST /api/training/submit` - Submit training job
- `POST /api/training/status` - Get job status
- `POST /api/training/cancel` - Cancel job
- `POST /api/model/download` - Download trained model
- `POST /api/dataset/upload` - Upload dataset
- `POST /api/resources/gpu` - Get GPU info

**Links:**
- [K'UHUL Documentation](https://github.com/cannaseedus-bot/XJSON-BOT)
- [GitHub Repository](https://github.com/cannaseedus-bot/XJSON-BOT)
- [Report Issues](https://github.com/cannaseedus-bot/XJSON-BOT/issues)
